# Medical IE V5.4 — PhoBERT + ViHealthBERT + GLiNER + Qwen end-to-end (Kaggle 20 GB safe)

Notebook chạy từ chuẩn bị dữ liệu, train ViHealthBERT/GLiNER/PhoBERT, assertion/linking, tải Qwen và full-100 inference.


## 1. Copy dữ liệu từ Kaggle Input

Cell idempotent: nếu repo/llama đã có trong Working thì không copy lại.

In [ ]:
%%bash
set -euo pipefail

REPO_INPUT="/kaggle/input/datasets/uyendungthanh/medical-ie-round2-hybrid-v5-4"
LLAMA_INPUT="/kaggle/input/datasets/uyendungthanh/llama-cpp-colab-cuda-build-tar-gz"

if [ ! -d "$REPO_INPUT" ]; then
  echo "Không thấy repo input: $REPO_INPUT" >&2
  exit 1
fi
if [ ! -d "$LLAMA_INPUT" ]; then
  echo "Không thấy llama input: $LLAMA_INPUT" >&2
  exit 1
fi

if [ ! -d /kaggle/working/medical_ie_round2_hybrid_v5_4 ]; then
  cp -a "$REPO_INPUT"/. /kaggle/working/
else
  echo "Repo đã có trong Working — bỏ qua copy."
fi

mkdir -p /kaggle/working/medical_ie_local_assets
if [ ! -d /kaggle/working/medical_ie_local_assets/llama.cpp ]; then
  cp -a "$LLAMA_INPUT"/. /kaggle/working/medical_ie_local_assets/
else
  echo "llama.cpp đã có trong Working — bỏ qua copy."
fi

mkdir -p /kaggle/working/v5_1_runs
find /kaggle/working -maxdepth 3 -name VERSION -print
find /kaggle/working/medical_ie_local_assets -path '*/build/bin/llama-server' -print

## 2. Khởi tạo đường dẫn

Tự nhận diện repo kể cả khi dataset có thêm một lớp thư mục.

In [ ]:
import os
from pathlib import Path

repo_candidates = [
    Path("/kaggle/working/medical_ie_round2_hybrid_v5_4"),
    Path("/kaggle/working/medical_ie_round2_hybrid_v5_4/medical_ie_round2_hybrid_v5_4"),
]
REPO = next((p for p in repo_candidates if (p / "configs/hybrid_round2_v5_4_phobert_consensus.yaml").exists()), None)
if REPO is None:
    found = list(Path("/kaggle/working").glob("**/configs/hybrid_round2_v5_4_phobert_consensus.yaml"))
    if not found:
        raise FileNotFoundError("Không tìm thấy repo V5.4 trong /kaggle/working")
    REPO = found[0].parents[1]

RUN_ROOT = Path("/kaggle/working/v5_1_runs")
ASSET_DIR = Path("/kaggle/working/medical_ie_local_assets")
LLAMA_LIB_DIR = ASSET_DIR / "llama.cpp/build/bin"
LLAMA_BIN = LLAMA_LIB_DIR / "llama-server"
HF_HOME = Path("/kaggle/working/hf_cache")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)

for key, value in {
    "REPO": REPO,
    "RUN_ROOT": RUN_ROOT,
    "ASSET_DIR": ASSET_DIR,
    "LLAMA_LIB_DIR": LLAMA_LIB_DIR,
    "LLAMA_BIN": LLAMA_BIN,
    "HF_HOME": HF_HOME,
}.items():
    os.environ[key] = str(value)

os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["LD_LIBRARY_PATH"] = ":".join(dict.fromkeys([
    str(LLAMA_LIB_DIR),
    *[x for x in os.environ.get("LD_LIBRARY_PATH", "").split(":") if x],
]))

print("REPO:", REPO)
print("VERSION:", (REPO / "VERSION").read_text().strip())
print("RUN_ROOT:", RUN_ROOT)
print("LLAMA_BIN:", LLAMA_BIN)

In [ ]:
%cd $REPO

## 3. Cài dependency

`requirements-v5.txt` không chứa GLiNER/FlagEmbedding; vì vậy cài đồng thời V4 và V5 để có đầy đủ pipeline và pin `scikit-learn==1.8.0`.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq git git-lfs cmake build-essential libcurl4-openssl-dev pigz zip
!git lfs install

!python -m pip install --no-cache-dir --upgrade pip wheel "setuptools<70" jedi
!python -m pip install --no-cache-dir seqeval==1.2.2 --no-build-isolation
!python -m pip uninstall -y gradio gradio-client
!python -m pip install --no-cache-dir -r requirements-v4.txt -r requirements-v5.txt
!python -m pip install --no-cache-dir "setuptools<70"
!python -m pip check || true

### BẮT BUỘC: Restart Session một lần

Sau cell cài package, chọn **Run → Restart Session**. Sau restart, tiếp tục từ cell **4. Khởi tạo lại môi trường**. Không copy repo và không cài package lại.

## 4. Khởi tạo lại môi trường sau Restart

In [ ]:
import os
from pathlib import Path

found = list(Path("/kaggle/working").glob("**/configs/hybrid_round2_v5_4_phobert_consensus.yaml"))
if not found:
    raise FileNotFoundError("Không tìm thấy repo V5.4")
REPO = found[0].parents[1]
RUN_ROOT = Path("/kaggle/working/v5_1_runs")
ASSET_DIR = Path("/kaggle/working/medical_ie_local_assets")
LLAMA_LIB_DIR = ASSET_DIR / "llama.cpp/build/bin"
LLAMA_BIN = LLAMA_LIB_DIR / "llama-server"
HF_HOME = Path("/kaggle/working/hf_cache")

for key, value in {
    "REPO": REPO,
    "RUN_ROOT": RUN_ROOT,
    "ASSET_DIR": ASSET_DIR,
    "LLAMA_LIB_DIR": LLAMA_LIB_DIR,
    "LLAMA_BIN": LLAMA_BIN,
    "HF_HOME": HF_HOME,
}.items():
    os.environ[key] = str(value)

os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["LD_LIBRARY_PATH"] = ":".join(dict.fromkeys([
    str(LLAMA_LIB_DIR),
    *[x for x in os.environ.get("LD_LIBRARY_PATH", "").split(":") if x],
]))

print("REPO:", REPO)
print("VERSION:", (REPO / "VERSION").read_text().strip())

In [ ]:
%cd $REPO

In [ ]:
import sys, torch, transformers, sklearn, gliner
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("Transformers:", transformers.__version__)
print("scikit-learn:", sklearn.__version__)
print("GLiNER:", getattr(gliner, "__version__", "unknown"))
if sklearn.__version__ != "1.8.0":
    raise RuntimeError("Cần scikit-learn==1.8.0")

## 5. Công cụ giám sát dung lượng

Chạy lại cell này sau mỗi stage. `assert_free_gib()` sẽ dừng trước bước nặng nếu ổ quá đầy.

In [ ]:
import shutil
from pathlib import Path

def disk_report(depth=2):
    usage = shutil.disk_usage("/kaggle/working")
    print(
        f"Working: used={usage.used/1024**3:.2f} GiB | "
        f"free={usage.free/1024**3:.2f} GiB | total={usage.total/1024**3:.2f} GiB"
    )
    roots = []
    for path in Path("/kaggle/working").iterdir():
        try:
            size = sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) if path.is_dir() else path.stat().st_size
            roots.append((size, path))
        except OSError:
            pass
    for size, path in sorted(roots, reverse=True)[:12]:
        print(f"{size/1024**3:7.2f} GiB  {path}")

def assert_free_gib(minimum):
    free = shutil.disk_usage("/kaggle/working").free / 1024**3
    print(f"Free: {free:.2f} GiB; required: {minimum:.2f} GiB")
    if free < minimum:
        raise RuntimeError("Không đủ dung lượng; dừng trước khi tạo checkpoint mới")

disk_report()

## 6. Sửa shared-library symlink của llama.cpp

Kaggle có thể biến symlink thành file rỗng 0 byte. Cell chỉ sửa khi file version thật tồn tại và là ELF.

In [ ]:
%%bash
set -euo pipefail
LIB_DIR="$LLAMA_LIB_DIR"
cd "$LIB_DIR"
chmod +x llama-server

repair() {
  local lib="$1" major="$2" real="$3"
  [ -f "$real" ] || return 0
  file "$real" | grep -q ELF || { echo "File thật không hợp lệ: $real"; exit 1; }
  rm -f "$lib" "$major"
  ln -s "$real" "$major"
  ln -s "$major" "$lib"
  echo "repaired $lib -> $major -> $real"
}

repair libllama-common.so libllama-common.so.0 libllama-common.so.0.0.1
repair libllama.so        libllama.so.0        libllama.so.0.0.1
repair libmtmd.so         libmtmd.so.0         libmtmd.so.0.0.1
repair libggml.so         libggml.so.0         libggml.so.0.17.0
repair libggml-base.so    libggml-base.so.0    libggml-base.so.0.17.0
repair libggml-cpu.so     libggml-cpu.so.0     libggml-cpu.so.0.17.0
repair libggml-cuda.so    libggml-cuda.so.0    libggml-cuda.so.0.17.0

LD_LIBRARY_PATH="$LIB_DIR:${LD_LIBRARY_PATH:-}" ldd "$LLAMA_BIN" | grep -E 'not found|file too short' && exit 1 || true
LD_LIBRARY_PATH="$LIB_DIR:${LD_LIBRARY_PATH:-}" "$LLAMA_BIN" --version

## 7. Xác minh lineage V5.1 và build knowledge

Verifier phải giữ nguyên byte-for-byte so với V5.1. Repo V5.4 đã sửa runner ở mức hạ tầng để nhận model alias; không sửa logic pipeline.


In [ ]:
from pathlib import Path
import hashlib

verifier = Path("src/round2/llm_verifier.py")
digest = hashlib.sha256(verifier.read_bytes()).hexdigest()
expected = "206832a89ccefcb25c1beb81c80d103954c91d9258cc596398591f55f3b0ec58"
print("Verifier SHA-256:", digest)
assert digest == expected, "Qwen verifier không còn giống V5.1"
assert Path("configs/hybrid_round2_v5_4_phobert_consensus.yaml").is_file()
print("Lineage V5.1 verifier: OK")


In [ ]:
!python -m py_compile scripts/75_run_round2_local.py scripts/68_download_hybrid_models.py src/round2/llm_verifier.py


In [ ]:
!python scripts/103_build_v5_knowledge.py

In [ ]:
!pytest -q

In [ ]:
!python -m compileall -q src scripts tests && echo 'compileall OK'

# PHẦN A — ViHealthBERT curriculum 3 stage

## 8. Chỉ tải ViHealthBERT base

Chưa tải Qwen, GLiNER, BGE-M3 hay reranker ở bước này.

In [ ]:
assert_free_gib(5.0)

In [ ]:
!python scripts/68_download_hybrid_models.py \
  --root "$REPO" \
  --skip-e5 \
  --skip-qwen

In [ ]:
!find models/pretrained/vihealthbert-base-syllable -maxdepth 2 -type f \
  \( -name '*.safetensors' -o -name 'pytorch_model*.bin' \) -size +1M -print
!du -sh models/pretrained/vihealthbert-base-syllable

## 9. Chuẩn bị manifest và curriculum

In [ ]:
!python scripts/80_prepare_real100_manifest.py

In [ ]:
!python scripts/90_prepare_v4_training_data.py \
  --case-specs data/external_synthetic_v4/case_specs.jsonl \
  --marked-notes data/external_synthetic_v4/marked_notes.jsonl \
  --output-dir data/processed/v4_curriculum \
  --seed 42 \
  --mixed-synthetic-limit 150 \
  --real-repeat 3
!cat data/processed/v4_curriculum/stats.json

## 10. Stage 1 — synthetic warm-up

Train 1 epoch từ ViHealthBERT base. Output stage 1 được xác minh trước khi xóa baseline.

In [ ]:
assert_free_gib(4.5)

In [ ]:
!rm -rf models/ner/vihealthbert-round2-v4-curriculum-stage1
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python scripts/82_train_vihealthbert_real100.py \
  --config configs/ner_round2_v4_curriculum.yaml \
  --real-manifest data/processed/v4_curriculum/stage1_synthetic/manifest.jsonl \
  --pretrained-dir models/pretrained/vihealthbert-base-syllable \
  --output-dir models/ner/vihealthbert-round2-v4-curriculum-stage1 \
  --epochs 1 \
  --learning-rate 1.2e-5

In [ ]:
%%bash
set -euo pipefail
OUT="models/ner/vihealthbert-round2-v4-curriculum-stage1"
test -s "$OUT/training_summary.json"
WEIGHT="$(find "$OUT" -maxdepth 1 -type f \( -name '*.safetensors' -o -name 'pytorch_model*.bin' \) -size +1M -print -quit)"
test -n "$WEIGHT"
echo "Stage 1 valid: $WEIGHT"
find "$OUT" -maxdepth 1 -type d -name 'checkpoint-*' -exec rm -rf {} +
rm -rf models/pretrained/vihealthbert-base-syllable
sync
du -sh "$OUT"
df -h /kaggle/working

## 11. Stage 2 — real-dominant mixed

Train 2 epoch từ stage 1. Sau khi xác minh stage 2, xóa stage 1.

In [ ]:
assert_free_gib(4.5)

In [ ]:
!rm -rf models/ner/vihealthbert-round2-v4-curriculum-stage2
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python scripts/82_train_vihealthbert_real100.py \
  --config configs/ner_round2_v4_curriculum.yaml \
  --real-manifest data/processed/v4_curriculum/stage2_mixed/manifest.jsonl \
  --pretrained-dir models/ner/vihealthbert-round2-v4-curriculum-stage1 \
  --output-dir models/ner/vihealthbert-round2-v4-curriculum-stage2 \
  --epochs 2 \
  --learning-rate 8e-6

In [ ]:
%%bash
set -euo pipefail
OUT="models/ner/vihealthbert-round2-v4-curriculum-stage2"
test -s "$OUT/training_summary.json"
WEIGHT="$(find "$OUT" -maxdepth 1 -type f \( -name '*.safetensors' -o -name 'pytorch_model*.bin' \) -size +1M -print -quit)"
test -n "$WEIGHT"
echo "Stage 2 valid: $WEIGHT"
find "$OUT" -maxdepth 1 -type d -name 'checkpoint-*' -exec rm -rf {} +
rm -rf models/ner/vihealthbert-round2-v4-curriculum-stage1
sync
du -sh "$OUT"
df -h /kaggle/working

## 12. Stage 3 — real-only adaptation

Train 3 epoch từ stage 2. Sau khi xác minh model cuối, xóa stage 2 và checkpoint cuối.

In [ ]:
assert_free_gib(5.0)

In [ ]:
!rm -rf models/ner/vihealthbert-round2-v4-curriculum
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python scripts/82_train_vihealthbert_real100.py \
  --config configs/ner_round2_v4_curriculum.yaml \
  --real-manifest data/processed/v4_curriculum/stage3_real/manifest.jsonl \
  --pretrained-dir models/ner/vihealthbert-round2-v4-curriculum-stage2 \
  --output-dir models/ner/vihealthbert-round2-v4-curriculum \
  --epochs 3 \
  --learning-rate 5e-6

In [ ]:
%%bash
set -euo pipefail
FINAL="models/ner/vihealthbert-round2-v4-curriculum"
test -s "$FINAL/training_summary.json"
WEIGHT="$(find "$FINAL" -maxdepth 1 -type f \( -name '*.safetensors' -o -name 'pytorch_model*.bin' \) -size +1M -print -quit)"
test -n "$WEIGHT"
echo "Final ViHealthBERT valid: $WEIGHT"
find "$FINAL" -maxdepth 1 -type d -name 'checkpoint-*' -exec rm -rf {} +
rm -rf models/ner/vihealthbert-round2-v4-curriculum-stage2
sync
du -sh "$FINAL"
df -h /kaggle/working

# PHẦN B — GLiNER V5.4 chunked training

## 13. Chỉ tải GLiNER base

Không dùng `scripts/96_download_v4_models.py` ở đây vì script đó tải cả BGE-M3 và reranker cùng lúc.

In [ ]:
assert_free_gib(7.0)

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

dest = Path("models/gliner/gliner_multi-v2.1")
if not dest.exists() or not any(dest.rglob("*.safetensors")):
    snapshot_download(
        repo_id="urchade/gliner_multi-v2.1",
        local_dir=dest,
        local_dir_use_symlinks=False,
    )
print("GLiNER base:", dest)

In [ ]:
!du -sh models/gliner/gliner_multi-v2.1 && df -h /kaggle/working

## 14. Chuẩn bị dataset chunked tích hợp V5.4

Script repo tự chunk 220 token, overlap 64, thêm rescue windows, giữ toàn bộ entity và loại chunk rỗng.

In [ ]:
!rm -rf data/processed/gliner_v5_1_chunked_positive
!python scripts/92_prepare_gliner_v4_dataset.py \
  --train-manifest data/processed/v4_curriculum/stage2_mixed/manifest.jsonl \
  --dev-manifest data/processed/v4_curriculum/dev_real/manifest.jsonl \
  --output-dir data/processed/gliner_v5_1_chunked_positive \
  --chunk-size 220 \
  --overlap 64
!cat data/processed/gliner_v5_1_chunked_positive/chunk_stats.json

In [ ]:
!python scripts/93_train_gliner_v4.py \
  --config configs/gliner_v5_2_train_batch4_1800.yaml \
  --validate-only

## 15. Fine-tune GLiNER — batch 4, 1.800 steps

Profile chính: physical batch 4 × accumulation 4 = effective batch 16. Số step tăng từ 1.200 lên 1.800. Nếu T4 báo CUDA OOM, chạy fallback batch 2 × accumulation 8 ở cell kế tiếp. Không chạy cả hai profile khi batch 4 đã thành công.


In [ ]:
assert_free_gib(6.5)

In [ ]:
!pkill -f llama-server || true
!pkill -f 93_train_gliner_v4.py || true
!rm -rf models/gliner/medical-ie-v5-2
!nvidia-smi

In [ ]:
!CUDA_VISIBLE_DEVICES=0 \
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
TOKENIZERS_PARALLELISM=false \
OMP_NUM_THREADS=2 \
python scripts/93_train_gliner_v4.py \
  --config configs/gliner_v5_2_train_batch4_1800.yaml

### Fallback chỉ khi batch 4 báo CUDA OOM

Chỉ chạy cell kế tiếp nếu batch-4 thất bại. Fallback vẫn giữ effective batch 16 và 1.800 steps.


In [ ]:
# CHỈ CHẠY NẾU PROFILE BATCH 4 ĐÃ OOM
!rm -rf models/gliner/medical-ie-v5-2
!CUDA_VISIBLE_DEVICES=0 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TOKENIZERS_PARALLELISM=false OMP_NUM_THREADS=2 python -u scripts/93_train_gliner_v4.py --config configs/gliner_v5_2_train_batch2_1800_fallback.yaml


## 16. Xác minh GLiNER tuned rồi xóa base/checkpoint

In [ ]:
%%bash
set -euo pipefail
TUNED="models/gliner/medical-ie-v5-2"
test -s "$TUNED/training_summary.json"
WEIGHT="$(find "$TUNED" -type f \( -name '*.safetensors' -o -name 'pytorch_model*.bin' \) -not -path '*/checkpoint-*/*' -size +1M -print -quit)"
test -n "$WEIGHT"
echo "GLiNER tuned weight: $WEIGHT"

In [ ]:
!HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 CUDA_VISIBLE_DEVICES='' python - <<'PY'
from gliner import GLiNER
model = GLiNER.from_pretrained('models/gliner/medical-ie-v5-2', local_files_only=True)
print('GLiNER tuned offline load: OK')
del model
PY

In [ ]:
%%bash
set -euo pipefail
TUNED="models/gliner/medical-ie-v5-2"
test -s "$TUNED/training_summary.json"
find "$TUNED" -type d -name 'checkpoint-*' -exec rm -rf {} +
rm -rf models/gliner/gliner_multi-v2.1
sync
du -sh "$TUNED"
df -h /kaggle/working

# PHẦN C — Assertion, BGE-M3, reranker và indexes

## 17. Train lại assertion scope với scikit-learn 1.8.0

In [ ]:
!rm -f models/assertion_scope/v4_scope.joblib
!python scripts/94_prepare_assertion_scope_data.py
!python scripts/95_train_assertion_scope_classifier.py \
  --input data/processed/assertion_scope_v4.jsonl \
  --output models/assertion_scope/v4_scope.joblib

In [ ]:
!python - <<'PY'
import joblib, sklearn
p='models/assertion_scope/v4_scope.joblib'
a=joblib.load(p)
print('sklearn', sklearn.__version__)
print('labels', a['labels'])
print('micro_f1', a.get('micro_f1'))
print('assertion artifact OK')
PY

## 18. Tải BGE-M3 và reranker, bỏ ONNX/OpenVINO

Các thư mục ONNX của BGE-M3 từng chiếm khoảng 2.2 GB nhưng pipeline này dùng PyTorch/Safetensors.

In [ ]:
assert_free_gib(7.0)

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

ignore = [
    "onnx/*", "onnx/**", "openvino/*", "openvino/**",
    "*.onnx", "*.onnx_data", "*.xml", "*.blob",
]
models = [
    ("BAAI/bge-m3", Path("models/embeddings/bge-m3")),
    ("BAAI/bge-reranker-v2-m3", Path("models/rerankers/bge-reranker-v2-m3")),
]
for repo_id, dest in models:
    if not dest.exists() or not any(dest.rglob("*.safetensors")):
        snapshot_download(
            repo_id=repo_id,
            local_dir=dest,
            local_dir_use_symlinks=False,
            ignore_patterns=ignore,
        )
    print(repo_id, "->", dest)

In [ ]:
!du -sh models/embeddings/bge-m3 models/rerankers/bge-reranker-v2-m3
!find models/embeddings/bge-m3 -maxdepth 2 -type d -name onnx -print
!df -h /kaggle/working

## 19. Build ICD/RxNorm embedding indexes

In [ ]:
assert_free_gib(2.0)

In [ ]:
!pkill -f llama-server || true
!python scripts/97_build_v4_embedding_indexes.py \
  --config configs/hybrid_round2_v5_4_phobert_consensus.yaml

In [ ]:
!ls -lh data/processed/icd_embeddings_bge_m3.npz data/processed/rxnorm_embeddings_bge_m3.npz
!df -h /kaggle/working

# PHẦN C2 — PhoBERT-base-v2 độc lập cho V5.4

PhoBERT là NER model thứ ba, độc lập với ViHealthBERT và GLiNER. Primary config chỉ nhận PhoBERT entity có exact corroboration để tránh union recall không kiểm soát.


In [ ]:
# Tải PhoBERT-base-v2 vào local model store.
from huggingface_hub import snapshot_download
PHOBERT_BASE=REPO/'models/pretrained/phobert-base-v2'
if not PHOBERT_BASE.exists():
 snapshot_download(repo_id='vinai/phobert-base-v2',local_dir=str(PHOBERT_BASE),local_dir_use_symlinks=False)
assert (PHOBERT_BASE/'config.json').is_file();print(PHOBERT_BASE)


In [ ]:
# Curriculum dùng lại dữ liệu đã chuẩn bị ở Phần A: synthetic -> real-dominant mixed -> real-only.
assert_free_gib(5.0)
ph_env={**env,'CUDA_VISIBLE_DEVICES':'0','PYTHONPATH':str(REPO),'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
subprocess.run(['python','-u','scripts/110_train_phobert_v54_curriculum.py','--config','configs/ner_round2_v5_4_phobert.yaml'],cwd=REPO,env=ph_env,check=True)


In [ ]:
# Xác minh final PhoBERT; stage1/stage2 đã tự dọn để tiết kiệm disk.
PHOBERT_FINAL=REPO/'models/ner/phobert-round2-v5-4-curriculum'
assert (PHOBERT_FINAL/'config.json').is_file()
assert list(PHOBERT_FINAL.glob('*.safetensors')) or (PHOBERT_FINAL/'pytorch_model.bin').is_file()
assert (PHOBERT_FINAL/'training_summary.json').is_file()
print((PHOBERT_FINAL/'training_summary.json').read_text(encoding='utf-8')[:3000]);assert_free_gib(3.5)


## Config V5.4

- `hybrid_round2_v5_4_control_no_phobert.yaml`: control tương đương V5.3.1.
- `hybrid_round2_v5_4_phobert_consensus.yaml`: primary, PhoBERT exact-consensus.
- `hybrid_round2_v5_4_phobert_high_confidence.yaml`: ablation cho entity PhoBERT độc lập confidence ≥0.985.

Chạy control và consensus trước; chỉ chạy high-confidence sau khi hai bản đầu hoàn tất.


# PHẦN D — Qwen, kiểm tra asset và inference

## 20. Tải Qwen2.5-7B-Instruct Q4_K_M cuối cùng

Model khoảng 4,7 GB, lớn hơn Qwen3-4B. Chỉ tải sau khi đã xóa base/checkpoint GLiNER và ViHealthBERT trung gian. Không tải đồng thời Qwen3 control ở lần chạy chính.


In [ ]:
assert_free_gib(3.5)

In [ ]:
assert_free_gib(5.5)
!python scripts/68_download_hybrid_models.py \
  --root "$REPO" \
  --skip-vihealthbert \
  --skip-e5 \
  --qwen-repo bartowski/Qwen2.5-7B-Instruct-GGUF \
  --qwen-file Qwen2.5-7B-Instruct-Q4_K_M.gguf \
  --qwen-dir models/gguf/qwen2.5-7b
!ls -lh models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf


## 21. Kiểm tra inference assets

Không dùng `--mode all` vì base ViHealthBERT/GLiNER đã chủ động xóa sau khi fine-tune.

In [ ]:
!python scripts/102_check_v4_assets.py \
  --root "$REPO" \
  --mode inference

In [ ]:
%%bash
set -euo pipefail
test -x "$LLAMA_BIN"
test -s models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf
test -d models/embeddings/bge-m3
test -d models/rerankers/bge-reranker-v2-m3
test -s data/processed/icd10_byt_v5.csv
test -s data/external/rxnorm_lookup_flat.json
LD_LIBRARY_PATH="$LLAMA_LIB_DIR:${LD_LIBRARY_PATH:-}" "$LLAMA_BIN" --version
echo "All runtime assets OK"

In [ ]:
!grep -n -A6 '^vihealthbert:' configs/hybrid_round2_v5_4_phobert_consensus.yaml
!grep -n -A20 '^gliner:' configs/hybrid_round2_v5_4_phobert_consensus.yaml
!grep -n -A6 '^reranker:' configs/hybrid_round2_v5_4_phobert_consensus.yaml
!df -h /kaggle/working

## 22. Audit parser V5.4

In [ ]:
!python scripts/98_audit_round2_v4_structure.py \
  --input-dir data/raw/input_round_2 \
  --output "$RUN_ROOT/parser_audit_v5_1.json"

## 23. Smoke test full hybrid 5 file

Bắt đầu `gpu-layers=20` theo guide V5.4 vì Qwen + GLiNER + ViHealthBERT cùng dùng VRAM.

In [ ]:
!pkill -f 'llama-server.*8081' || true
!LD_LIBRARY_PATH="$LLAMA_LIB_DIR:${LD_LIBRARY_PATH:-}" \
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
TOKENIZERS_PARALLELISM=false \
python scripts/75_run_round2_local.py \
  --llama-bin "$LLAMA_BIN" \
  --qwen-model models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf \
  --config configs/hybrid_round2_v5_4_phobert_consensus.yaml \
  --input-dir data/raw/input_round_2 \
  --baseline-prediction-dir data/baselines/vihealthbert_27_6152 \
  --output-dir "$RUN_ROOT/full_hybrid_test5/predictions" \
  --debug-dir "$RUN_ROOT/full_hybrid_test5/debug" \
  --limit 5 \
  --overwrite \
  --gpu-layers 20 \
  --threads 4

In [ ]:
!find "$RUN_ROOT/full_hybrid_test5/predictions" -maxdepth 1 -name '*.json' | wc -l
!find "$RUN_ROOT/full_hybrid_test5/debug" -maxdepth 1 -name '*.debug.json' | wc -l

## 24. Chạy 100 file với resume/retry

Không dùng `--overwrite`: nếu server lỗi giữa chừng, lần thử sau bỏ qua file đã có đủ prediction và debug.

In [ ]:
%%bash
set +e
STATUS=1
for ATTEMPT in 1 2 3 4; do
  echo "===== ATTEMPT $ATTEMPT ====="
  pkill -f 'llama-server.*8081' 2>/dev/null || true
  sleep 3

  LD_LIBRARY_PATH="$LLAMA_LIB_DIR:${LD_LIBRARY_PATH:-}" \
  PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  TOKENIZERS_PARALLELISM=false \
  python scripts/75_run_round2_local.py \
    --llama-bin "$LLAMA_BIN" \
    --qwen-model models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf \
    --config configs/hybrid_round2_v5_4_phobert_consensus.yaml \
    --input-dir data/raw/input_round_2 \
    --baseline-prediction-dir data/baselines/vihealthbert_27_6152 \
    --output-dir "$RUN_ROOT/full_hybrid/predictions" \
    --debug-dir "$RUN_ROOT/full_hybrid/debug" \
    --limit 100 \
    --gpu-layers 20 \
    --threads 4

  STATUS=$?
  [ $STATUS -eq 0 ] && break
  sleep 5
done

exit $STATUS

## 25. Kiểm tra đủ 100 kết quả

In [ ]:
from pathlib import Path
import os

run_root = Path(os.environ["RUN_ROOT"]) / "full_hybrid"
pred = list((run_root / "predictions").glob("*.json"))
debug = list((run_root / "debug").glob("*.debug.json"))
print("predictions:", len(pred))
print("debug:", len(debug))
assert len(pred) == 100, "Chưa đủ 100 prediction"
assert len(debug) == 100, "Chưa đủ 100 debug"
print("✅ Đủ 100/100")

## 26. ZIP predictions và debug để tải về

Chỉ nén kết quả, không nén models để tránh nhân đôi nhiều GB trong Working.

In [ ]:
%%bash
set -euo pipefail
cd "$RUN_ROOT/full_hybrid"
rm -f /kaggle/working/v5_3_predictions.zip /kaggle/working/v5_3_debug.zip
zip -r -1 /kaggle/working/v5_3_predictions.zip predictions
zip -r -1 /kaggle/working/v5_3_debug.zip debug
ls -lh /kaggle/working/v5_3_predictions.zip /kaggle/working/v5_3_debug.zip

In [ ]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/v5_3_predictions.zip"))
display(FileLink("/kaggle/working/v5_3_debug.zip"))

## 27. Báo cáo dung lượng cuối

Để lưu model qua session khác, dùng **Save Version → Save output files**, sau đó tạo Kaggle Dataset từ output. Không tạo tar model trong cùng Working vì sẽ nhân đôi dung lượng và dễ vượt 20 GB.

In [ ]:
disk_report()

In [ ]:
!du -sh \
  models/ner/vihealthbert-round2-v4-curriculum \
  models/gliner/medical-ie-v5-2 \
  models/embeddings/bge-m3 \
  models/rerankers/bge-reranker-v2-m3 \
  models/gguf/qwen3-4b \
  data/processed/*bge_m3.npz \
  "$RUN_ROOT/full_hybrid" 2>/dev/null

# Checklist cuối

- Repo `VERSION` là `5.3.0`.
- `src/round2/llm_verifier.py` SHA-256 giống V5.1.
- GLiNER V5.4 có `training_summary.json`, batch 4/accumulation 4/1.800 steps; hoặc fallback batch 2/accumulation 8.
- Qwen2.5-7B Q4_K_M tồn tại và Qwen3 không được tải trong run chính.
- Config inference là `configs/hybrid_round2_v5_4_phobert_consensus.yaml`.
- Đủ 100 predictions/debug, position validation đạt.
- Luôn giữ submission V5.1 score 30.2386 làm control.


# PHẦN E — Xác minh cleanup V5.4

In [ ]:
from src.common.schema import Entity
from src.postprocessing.precision_cleanup_v53 import PrecisionCleanupV53, PrecisionCleanupV53Config

def _e(text, typ, source='gliner'):
    return Entity(text=text,start=0,end=len(text),type=typ,source=source,confidence=.99)
cleanup=PrecisionCleanupV53(PrecisionCleanupV53Config())
rows,report=cleanup.apply([_e('Xông khí dung','THUỐC'),_e('Xông khí dung','THUỐC','baseline_anchor'),_e('sỏi mật','TRIỆU_CHỨNG')])
assert any(e.source=='baseline_anchor' for e in rows)
assert any(e.text=='sỏi mật' and e.type=='CHẨN_ĐOÁN' for e in rows)
assert not any(e.text=='Xông khí dung' and e.source=='gliner' for e in rows)
print('V5.4 cleanup regression: PASSED')


# PHẦN F — Full-100 ablations

Chạy tuần tự, không chạy song song. `no_cleanup_control` phải gần tái lập kiến trúc score 31.1437; primary chỉ khác cleanup hẹp. Selective verifier chỉ kiểm Qwen/ViHealth và bypass GLiNER.

In [ ]:
import os,subprocess,time,shutil,json
from pathlib import Path
from IPython.display import FileLink,display

ABLATION_ROOT=Path('/kaggle/working/v5_3_ablations');ABLATION_ROOT.mkdir(parents=True,exist_ok=True)
ABLATIONS={
 'no_cleanup_control':'configs/hybrid_round2_v5_3_no_cleanup_control.yaml',
 'primary_cleanup':'configs/hybrid_round2_v5_4_phobert_consensus.yaml',
 'type_repair_only':'configs/hybrid_round2_v5_3_type_repair_only.yaml',
 'selective_verifier':'configs/hybrid_round2_v5_3_selective_verifier.yaml',
}
env=os.environ.copy();env.update({'PYTHONUNBUFFERED':'1','PYTHONPATH':str(REPO),'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','HF_DATASETS_OFFLINE':'1','TOKENIZERS_PARALLELISM':'false','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'})
env['LD_LIBRARY_PATH']=f"{LLAMA_LIB_DIR}:{env.get('LD_LIBRARY_PATH','')}"
print(ABLATIONS)


In [ ]:
def run_v53(name,config,attempts=4):
 root=ABLATION_ROOT/name;pred=root/'predictions';debug=root/'debug';logs=root/'logs'
 for p in [pred,debug,logs]:p.mkdir(parents=True,exist_ok=True)
 cmd=['python','-u','scripts/75_run_round2_local.py','--config',config,'--input-dir','data/raw/input_round_2','--output-dir',str(pred),'--debug-dir',str(debug),'--baseline-prediction-dir','data/baselines/vihealthbert_27_6152','--llama-bin',str(LLAMA_BIN),'--qwen-model','models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf','--model-alias','qwen2.5-7b','--gpu-layers','99','--threads','4','--server-timeout','600']
 for attempt in range(1,attempts+1):
  if len(list(pred.glob('*.json')))>=100:break
  subprocess.run(['pkill','-f','llama-server.*8081'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
  with (logs/f'attempt_{attempt}.log').open('a',encoding='utf-8',buffering=1) as log:
   p=subprocess.Popen(cmd,cwd=REPO,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
   for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
   p.wait()
  time.sleep(3)
 count=len(list(pred.glob('*.json')));assert count==100,(name,count)
 return root

for name,config in ABLATIONS.items():run_v53(name,config)


In [ ]:
# Structural summary before leaderboard submission.
from collections import Counter
import pandas as pd
rows=[]
for name in ABLATIONS:
 pred=ABLATION_ROOT/name/'predictions';debug=ABLATION_ROOT/name/'debug';entities=[];cleanup_counts=Counter()
 for p in pred.glob('*.json'):
  x=json.loads(p.read_text(encoding='utf-8'));entities+=x if isinstance(x,list) else x.get('entities',[])
 for p in debug.glob('*.json'):
  d=json.loads(p.read_text(encoding='utf-8'))
  for r in d.get('precision_cleanup_v53_report',[]) or []:cleanup_counts[(r.get('action'),r.get('reason'))]+=1
 elig=[e for e in entities if e.get('type') in {'CHẨN_ĐOÁN','THUỐC'}]
 rows.append({'variant':name,'files':len(list(pred.glob('*.json'))),'entities':len(entities),'diagnoses':sum(e.get('type')=='CHẨN_ĐOÁN' for e in entities),'drugs':sum(e.get('type')=='THUỐC' for e in entities),'candidate_coverage':sum(bool(e.get('candidates')) for e in elig)/len(elig) if elig else 0,'cleanup_drop':sum(v for (a,_),v in cleanup_counts.items() if a=='drop'),'cleanup_repair':sum(v for (a,_),v in cleanup_counts.items() if a=='repair')})
summary=pd.DataFrame(rows);display(summary);summary.to_csv(ABLATION_ROOT/'summary.csv',index=False)


In [ ]:
EXPORT=Path('/kaggle/working/v5_3_exports');EXPORT.mkdir(parents=True,exist_ok=True)
for name in ABLATIONS:
 root=ABLATION_ROOT/name
 for kind in ['predictions','debug']:
  base=EXPORT/f'v5_3_{name}_{kind}';target=base.with_suffix('.zip')
  if target.exists():target.unlink()
  shutil.make_archive(str(base),'zip',root_dir=root/kind);display(FileLink(str(target)))
 shutil.copy2(REPO/ABLATIONS[name],EXPORT/f'{name}.yaml')
shutil.copy2(ABLATION_ROOT/'summary.csv',EXPORT/'summary.csv')


# PHẦN G — GLiNER continuation real-dominant (tùy chọn)

Chỉ chạy sau khi đã lưu outputs của checkpoint V5.2. Nhánh này dùng checkpoint 1.800-step làm base, real train lặp 5 lần + tối đa 80 synthetic, continuation 400 steps với learning rate thấp. Đây là experiment, không thay primary mặc định trước khi có điểm leaderboard.

In [ ]:
# Chuẩn bị real-dominant positive-chunk mix.
subprocess.run(['python','scripts/108_prepare_v53_real_mix.py','--real-repeat','5','--synthetic-limit','80'],cwd=REPO,check=True,env=env)


In [ ]:
# Train continuation trên một GPU; cần còn đủ dung lượng.
assert_free_gib(4.0)
subprocess.run(['python','-u','scripts/93_train_gliner_v4.py','--config','configs/gliner_v5_3_continue_400.yaml'],cwd=REPO,check=True,env={**env,'CUDA_VISIBLE_DEVICES':'0'})


Sau khi continuation hoàn tất, chạy full-100 bằng `configs/hybrid_round2_v5_3_augmented_gliner.yaml` vào output riêng và so sánh với `no_cleanup_control`/`primary_cleanup`. Không chọn theo training loss.